ALIGNMENT CHECK FOR INDUSTRIES (ALL DATASETS)

In [1]:
# ===============================
# Industry Alignment Check (Final)
# ===============================
import sys
from pathlib import Path
import pandas as pd

# -------------------------------
# Fix Python path to find src/config.py
# -------------------------------
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.config import STANDARD_INDUSTRIES

# -------------------------------
# Load datasets
# -------------------------------
bls_df = pd.read_csv('../../insights-project-v2/data/final/bls_employment_by_industry.csv')
wvsos_df = pd.read_csv('../../insights-project-v2/data/final/wvsos_final_yearly.csv')
bea_df = pd.read_csv('../../insights-project-v2/data/final/bea_final.csv')

# -------------------------------
# Check function
# -------------------------------
def check_industries(df, dataset_name, allow_unknown=False):
    unique_industries = set(df['industry'].unique())

    # Handle Unknown Industry separately (allowed only for WVSOS)
    if allow_unknown and 'Unknown Industry' in unique_industries:
        unique_industries.remove('Unknown Industry')

    invalid = unique_industries - set(STANDARD_INDUSTRIES)
    missing = set(STANDARD_INDUSTRIES) - unique_industries

    print(f"\n--- {dataset_name} ---")
    print(f"Unique industries count: {len(unique_industries)}")

    if not invalid and not missing:
        print("✅ All industries match standard list")
    else:
        if invalid:
            print("❌ INVALID industries:", invalid)
        if missing:
            print("❌ MISSING industries:", missing)

    # Extra checks
    if 'Total Nonfarm' in df['industry'].values:
        print("❌ Contains 'Total Nonfarm' (should NOT be present)")
    
    if 'Unknown Industry' in df['industry'].values:
        print("ℹ️ Contains 'Unknown Industry' (allowed for WVSOS only)")

# -------------------------------
# Run checks
# -------------------------------
check_industries(bls_df, "BLS")
check_industries(wvsos_df, "WVSOS", allow_unknown=True)
check_industries(bea_df, "BEA")


--- BLS ---
Unique industries count: 13
✅ All industries match standard list

--- WVSOS ---
Unique industries count: 13
✅ All industries match standard list
ℹ️ Contains 'Unknown Industry' (allowed for WVSOS only)

--- BEA ---
Unique industries count: 13
✅ All industries match standard list


In [8]:
# ===============================
# FINAL SQL-READINESS CHECK
# ===============================
import pandas as pd

# -----------------------------
# Paths to transformed datasets
# -----------------------------
bls_employment_path = '../../insights-project-v2/data/final/bls_employment_by_industry.csv'
bls_unemployment_path = '../../insights-project-v2/data/final/bls_unemployment_summary.csv'
wvsos_path = '../../insights-project-v2/data/final/wvsos_final_yearly.csv'
bea_path = '../../insights-project-v2/data/final/bea_final.csv'

# -----------------------------
# Load datasets
# -----------------------------
bls_emp_df = pd.read_csv(bls_employment_path)  # columns: year, industry, avg_employment
bls_unemp_df = pd.read_csv(bls_unemployment_path)  # columns: year, unemployment_rate, labor_force_participation_rate, labor_force
wvsos_df = pd.read_csv(wvsos_path)  # columns: year, industry, new_filings, terminations
bea_df = pd.read_csv(bea_path)  # columns: year, industry, gdp

# -----------------------------
# Expected schema
# -----------------------------
BLS_EMP_COLUMNS = ['year', 'industry', 'avg_employment']
BLS_UNEMP_COLUMNS = ['year', 'unemployment_rate', 'labor_force_participation_rate', 'labor_force']
WVSOS_COLUMNS = ['year', 'industry', 'new_filings', 'terminations']
BEA_COLUMNS = ['year', 'industry', 'gdp']

STANDARD_INDUSTRIES = [
    'Mining and Logging', 'Construction', 'Manufacturing', 'Trade, Transportation, and Utilities',
    'Information', 'Finance and Insurance', 'Real Estate and Rental and Leasing',
    'Professional and Business Services', 'Private Educational Services',
    'Health Care and Social Assistance', 'Leisure and Hospitality', 'Other Services', 'Government'
]

# -----------------------------
# Function to check datasets
# -----------------------------
def check_dataset(df, expected_columns, dataset_name, allow_unknown=False):
    print(f"--- {dataset_name} ---")
    # Columns check
    actual_cols = list(df.columns)
    if actual_cols == expected_columns:
        print("✅ Columns match expected schema")
    else:
        print(f"❌ Columns mismatch. Actual: {actual_cols}")
    # Dtypes check
    print("Data types:")
    print(df.dtypes)
    # Industry check (only for datasets with industry)
    if 'industry' in df.columns:
        unique_industries = set(df['industry'].unique())
        invalid = unique_industries - set(STANDARD_INDUSTRIES)
        if allow_unknown:
            invalid.discard('Unknown Industry')
        missing = set(STANDARD_INDUSTRIES) - unique_industries
        if not invalid and not missing:
            print("✅ All industries match standard list")
        else:
            if invalid:
                print("❌ INVALID industries:", invalid)
            if missing:
                print("❌ MISSING industries:", missing)
    print("\n")

# -----------------------------
# Run checks on each dataset
# -----------------------------
check_dataset(bls_emp_df, BLS_EMP_COLUMNS, "BLS Employment")
check_dataset(bls_unemp_df, BLS_UNEMP_COLUMNS, "BLS Unemployment")
check_dataset(wvsos_df, WVSOS_COLUMNS, "WVSOS", allow_unknown=True)
check_dataset(bea_df, BEA_COLUMNS, "BEA GDP")

# -----------------------------
# Optional join readiness check
# -----------------------------
print("--- Join Readiness Check (year + industry) ---")
datasets_with_industry = [bls_emp_df, wvsos_df, bea_df]
for i, df1 in enumerate(datasets_with_industry):
    for j, df2 in enumerate(datasets_with_industry):
        if i < j:
            df1_pairs = set(zip(df1['year'], df1['industry']))
            df2_pairs = set(zip(df2['year'], df2['industry']))
            missing_in_df2 = df1_pairs - df2_pairs
            missing_in_df1 = df2_pairs - df1_pairs
            print(f"{df1.columns[1]} -> {df2.columns[1]} missing pairs: {len(missing_in_df2)}")
            print(f"{df2.columns[1]} -> {df1.columns[1]} missing pairs: {len(missing_in_df1)}")
print("✅ If all missing pairs = 0 or expected due to Unknown Industry, your datasets are SQL-ready")

--- BLS Employment ---
❌ Columns mismatch. Actual: ['industry', 'year', 'avg_employment']
Data types:
industry          object
year               int64
avg_employment     int64
dtype: object
✅ All industries match standard list


--- BLS Unemployment ---
✅ Columns match expected schema
Data types:
year                                int64
unemployment_rate                 float64
labor_force_participation_rate    float64
labor_force                       float64
dtype: object


--- WVSOS ---
✅ Columns match expected schema
Data types:
year             int64
industry        object
new_filings      int64
terminations     int64
dtype: object
✅ All industries match standard list


--- BEA GDP ---
✅ Columns match expected schema
Data types:
year          int64
industry     object
gdp         float64
dtype: object
✅ All industries match standard list


--- Join Readiness Check (year + industry) ---
year -> industry missing pairs: 0
industry -> year missing pairs: 20
year -> industry missing 